# Librerias 

In [7]:
import pandas as pd
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Iterable, Optional, Set
from copy import deepcopy


In [10]:
CSV_FILES = [
    r"D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad2\Year 2009-2010.csv",
    r"D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tareaunidad2\Year 2010-2011.csv",
]
# Parámetros
USE_DESCRIPTION = True    
MINSUP_ITEMSETS_REL   = 0.01   
MINSUP_SEQUENCES_REL  = 0.005  
MINSUP_ITEMSETS_ABS   = None   
MINSUP_SEQUENCES_ABS  = None   
MIN_ITEMSET_SIZE   = 1  # tamaño mínimo del itemset a reportar (>=1)
MIN_PATTERN_LENGTH = 1  # longitud mínima del patrón secuencial (>=1)

FILTER_COUNTRY = None          
FILTER_DATE_MIN = None         
FILTER_DATE_MAX = None          

# Limpieza 

In [11]:
def _get_any(cols_lower_to_orig: Dict[str, str], candidates: List[str]) -> str:
    for cand in candidates:
        if cand.lower() in cols_lower_to_orig:
            return cols_lower_to_orig[cand.lower()]
    raise KeyError(f"No encuentro ninguna de estas columnas: {candidates}")

def load_and_clean(csv_path: str, use_description: bool = True) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding="ISO-8859-1")
    cols_map = {c.lower(): c for c in df.columns}

    inv_col  = _get_any(cols_map, ["InvoiceNo", "Invoice"])
    qty_col  = _get_any(cols_map, ["Quantity"])
    stk_col  = _get_any(cols_map, ["StockCode"])
    desc_col = _get_any(cols_map, ["Description"])
    cust_col = _get_any(cols_map, ["CustomerID", "Customer ID"])
    date_col = _get_any(cols_map, ["InvoiceDate"])
    country_col = cols_map.get("country", None)

    # Quitar cancelaciones y cantidades <= 0
    mask_not_cancel = ~df[inv_col].astype(str).str.startswith(("C", "c"))
    mask_qty_pos = df[qty_col] > 0
    df = df[mask_not_cancel & mask_qty_pos].copy()

    # Tipos y NaNs
    df[cust_col] = pd.to_numeric(df[cust_col], errors="coerce")
    need_cols = [cust_col, stk_col, date_col]
    if use_description:
        need_cols.append(desc_col)
    df = df.dropna(subset=need_cols)

    # Limpiar textos
    df[stk_col] = df[stk_col].astype(str).str.strip()
    if use_description:
        df[desc_col] = df[desc_col].astype(str).str.strip()
    df[inv_col] = df[inv_col].astype(str).str.strip()

    # Parsear fecha
    try:
        df[date_col] = pd.to_datetime(df[date_col])
    except Exception:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")
        df = df.dropna(subset=[date_col])

    keep = [inv_col, stk_col, desc_col, cust_col, date_col]
    if country_col: keep.append(country_col)
    keep = [c for c in keep if c in df.columns]

    df = df[keep].rename(columns={
        inv_col: "InvoiceNo",
        stk_col: "StockCode",
        desc_col: "Description",
        cust_col: "CustomerID",
        date_col: "InvoiceDate",
        country_col: "Country" if country_col else None
    })
    return df

def load_many(csv_paths: List[str], use_description: bool = True) -> pd.DataFrame:
    frames = []
    for p in csv_paths:
        dfi = load_and_clean(p, use_description=use_description)
        dfi["__source"] = p  
        frames.append(dfi)
    df_all = pd.concat(frames, ignore_index=True)

    # Deduplicación exacta por claves transaccionales básicas
    dedup_keys = ["InvoiceNo", "StockCode", "Description", "CustomerID", "InvoiceDate"]
    dedup_keys = [k for k in dedup_keys if k in df_all.columns]
    df_all = df_all.drop_duplicates(subset=dedup_keys)

    # Orden cronológico
    if "InvoiceDate" in df_all.columns:
        df_all = df_all.sort_values("InvoiceDate").reset_index(drop=True)

    return df_all

# Cargar uno o varios
if len(CSV_FILES) == 1:
    df = load_and_clean(CSV_FILES[0], use_description=USE_DESCRIPTION)
    print("[INFO] Cargado 1 CSV:", df.shape)
else:
    df = load_many(CSV_FILES, use_description=USE_DESCRIPTION)
    print(f"[INFO] Cargados {len(CSV_FILES)} CSVs:", df.shape)

# Filtros opcionales
if FILTER_COUNTRY and ("Country" in df.columns):
    df = df[df["Country"] == FILTER_COUNTRY].copy()

if FILTER_DATE_MIN:
    df = df[df["InvoiceDate"] >= pd.to_datetime(FILTER_DATE_MIN)]
if FILTER_DATE_MAX:
    df = df[df["InvoiceDate"] < pd.to_datetime(FILTER_DATE_MAX)]

print("[INFO] Después de filtros:", df.shape)
display(df.head(5))

[INFO] Cargados 2 CSVs: (768935, 7)
[INFO] Después de filtros: (768935, 7)


,InvoiceNo,StockCode,Description,CustomerID,InvoiceDate,Country,__source
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,13085.0,2009-12-01 07:45:00,United Kingdom,D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tarea...
1,489434,79323P,PINK CHERRY LIGHTS,13085.0,2009-12-01 07:45:00,United Kingdom,D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tarea...
2,489434,79323W,WHITE CHERRY LIGHTS,13085.0,2009-12-01 07:45:00,United Kingdom,D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tarea...
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",13085.0,2009-12-01 07:45:00,United Kingdom,D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tarea...
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,13085.0,2009-12-01 07:45:00,United Kingdom,D:\Datases_CD\7mo\Datos_Masivos_2\Tareas\Tarea...


# Funciones auxiliares 

In [12]:
def build_baskets(df: pd.DataFrame, use_description: bool = True) -> List[Set[str]]:
    item_col = "Description" if use_description else "StockCode"
    baskets = (
        df.groupby("InvoiceNo")[item_col]
          .apply(lambda s: set(s.astype(str)))
          .tolist()
    )
    return [b for b in baskets if b]

def build_sequences(df: pd.DataFrame, use_description: bool = True) -> List[List[Set[str]]]:
    item_col = "Description" if use_description else "StockCode"
    inv_baskets = (
        df.groupby(["CustomerID", "InvoiceNo", "InvoiceDate"])[item_col]
          .apply(lambda s: set(s.astype(str)))
          .reset_index()
          .sort_values(["CustomerID", "InvoiceDate", "InvoiceNo"])
    )
    sequences: List[List[Set[str]]] = []
    for _, grp in inv_baskets.groupby("CustomerID"):
        seq = [set(items) for items in grp[item_col]]
        seq = [e for e in seq if e]
        if seq:
            sequences.append(seq)
    return sequences

baskets = build_baskets(df, USE_DESCRIPTION)
sequences = build_sequences(df, USE_DESCRIPTION)
print(f"[INFO] Baskets (facturas): {len(baskets):,} | Sequences (clientes): {len(sequences):,}")

[INFO] Baskets (facturas): 36,975 | Sequences (clientes): 5,881


In [13]:
def resolve_min_support_rel_abs(total: int, rel: Optional[float], abs_sup: Optional[int]) -> int:
    if abs_sup is not None:
        return max(1, int(abs_sup))
    if rel is not None:
        return max(1, int(rel * total + 1e-9))
    raise ValueError("Debes definir soporte relativo o absoluto.")

# Para itemsets (baskets)
MIN_SUP_ITEMSETS = resolve_min_support_rel_abs(len(baskets), MINSUP_ITEMSETS_REL, MINSUP_ITEMSETS_ABS)
# Para secuencias (clientes)
MIN_SUP_SEQUENCES = resolve_min_support_rel_abs(len(sequences), MINSUP_SEQUENCES_REL, MINSUP_SEQUENCES_ABS)

print(f"[INFO] min_sup itemsets (abs):  {MIN_SUP_ITEMSETS}")
print(f"[INFO] min_sup secuencias (abs): {MIN_SUP_SEQUENCES}")

[INFO] min_sup itemsets (abs):  369
[INFO] min_sup secuencias (abs): 29


# FP-Growth

In [14]:
def _build_flist(transactions: List[List[str]], min_support: int):
    freq = Counter()
    for t in transactions:
        freq.update(t)
    freq = Counter({i: c for i, c in freq.items() if c >= min_support})
    # Orden por soporte desc y luego por nombre para estabilidad
    order_list = [i for i, _ in sorted(freq.items(), key=lambda kv: (-kv[1], kv[0]))]
    return order_list, freq

def _insert_transaction(tree_root: dict, header: Dict[str, Dict], ordered_items: List[str]):
    """
    Nodo es un dict: {'item', 'count', 'parent', 'children', 'link'}
    header: item -> {'support': int, 'head': nodo o None}
    """
    node = tree_root
    for itm in ordered_items:
        if itm not in node['children']:
            child = {'item': itm, 'count': 0, 'parent': node, 'children': {}, 'link': None}
            node['children'][itm] = child
            if header[itm]['head'] is None:
                header[itm]['head'] = child
            else:
                cur = header[itm]['head']
                while cur['link'] is not None:
                    cur = cur['link']
                cur['link'] = child
        node = node['children'][itm]
        node['count'] += 1

def _build_fptree(transactions: List[List[str]], min_support: int):
    order_list, freq_counter = _build_flist(transactions, min_support)
    if not order_list:
        return None, None, None
    header = {i: {'support': freq_counter[i], 'head': None} for i in order_list}
    root = {'item': None, 'count': 0, 'parent': None, 'children': {}, 'link': None}
    for t in transactions:
        ordered = [i for i in order_list if i in t]
        if ordered:
            _insert_transaction(root, header, ordered)
    return root, header, order_list

def _is_single_path(root: dict) -> bool:
    node = root
    while True:
        if len(node['children']) > 1:
            return False
        if len(node['children']) == 0:
            return True
        node = next(iter(node['children'].values()))

def _conditional_pattern_base(header: Dict[str, Dict], item: str) -> List[List[str]]:
    patterns = []
    node = header[item]['head']
    while node is not None:
        path = []
        parent = node['parent']
        while parent and parent['item'] is not None:
            path.append(parent['item'])
            parent = parent['parent']
        if path:
            for _ in range(node['count']):
                patterns.append(path)
        node = node['link']
    return patterns

def fpgrowth(transactions: List[Iterable[str]], min_support_abs: int, min_itemset_size: int = 1):
    T = [list(set(t)) for t in transactions if t]
    if not T:
        return []
    root, header, _ = _build_fptree(T, min_support_abs)
    if root is None:
        return []

    results = []

    def mine(tree_root: dict, header_tbl: Dict[str, Dict], suffix: Tuple[str, ...]):
        if _is_single_path(tree_root):
            path_items = []
            node = tree_root
            while node['children']:
                node = next(iter(node['children'].values()))
                path_items.append((node['item'], node['count']))
            from itertools import combinations
            for r in range(1, len(path_items)+1):
                for combo in combinations(path_items, r):
                    items = tuple(sorted([i for i, _ in combo]))
                    sup = min(c for _, c in combo)
                    full = tuple(sorted(items + suffix))
                    if len(full) >= min_itemset_size:
                        results.append((full, sup))
            return

        items_sorted = sorted(header_tbl.items(), key=lambda kv: (kv[1]['support'], kv[0]))
        for item, meta in items_sorted:
            new_suffix = tuple(sorted((item,) + suffix))
            if len(new_suffix) >= min_itemset_size:
                results.append((new_suffix, meta['support']))

            cond_patterns = _conditional_pattern_base(header_tbl, item)
            cond_root, cond_header, _ = _build_fptree(cond_patterns, min_support_abs)
            if cond_root and cond_header:
                mine(cond_root, cond_header, new_suffix)

    mine(root, header, ())
    # deduplicar (máximo soporte)
    best = {}
    for items, sup in results:
        best[tuple(sorted(items))] = max(best.get(tuple(sorted(items)), 0), sup)
    out = [(k, v) for k, v in best.items()]
    out.sort(key=lambda x: (-x[1], x[0]))
    return out

itemsets = fpgrowth([list(b) for b in baskets], min_support_abs=MIN_SUP_ITEMSETS, min_itemset_size=MIN_ITEMSET_SIZE)
print(f"[INFO] Itemsets encontrados: {len(itemsets):,}")
display(pd.DataFrame(
    [{"itemset": " | ".join(items), "size": len(items), "support": sup}
     for items, sup in itemsets[:20]]
))


[INFO] Itemsets encontrados: 727


,itemset,size,support
0,WHITE HANGING HEART T-LIGHT HOLDER,1,4888
1,REGENCY CAKESTAND 3 TIER,1,3318
2,ASSORTED COLOUR BIRD ORNAMENT,1,2652
3,JUMBO BAG RED RETROSPOT,1,2612
4,PARTY BUNTING,1,2078
5,LUNCH BAG BLACK SKULL.,1,1997
6,LUNCH BAG SPACEBOY DESIGN,1,1874
7,REX CASH+CARRY JUMBO SHOPPER,1,1857
8,HOME BUILDING BLOCK WORD,1,1831
9,STRAWBERRY CERAMIC TRINKET BOX,1,1818


# PrefixSpan

In [15]:
def _is_subsequence(pat: List[Set[str]], seq: List[Set[str]]) -> bool:
    i = 0
    for event in seq:
        if pat[i].issubset(event):
            i += 1
            if i == len(pat):
                return True
    return False

def _project_db(db: List[List[Set[str]]], prefix: List[Set[str]], grow_in_same_event: bool):
    proj = []
    for seq in db:
        k = 0
        pos = -1
        for idx, event in enumerate(seq):
            if prefix[k].issubset(event):
                k += 1
                if k == len(prefix):
                    pos = idx
                    break
        if pos == -1:
            continue
        if grow_in_same_event:
            new_first = set(seq[pos]) - set(prefix[-1])
            suffix = [new_first] + [set(e) for e in seq[pos+1:]]
        else:
            suffix = [set(e) for e in seq[pos+1:]]
        if suffix and len(suffix[0]) == 0:
            suffix = suffix[1:]
        proj.append(suffix)
    return proj

def prefixspan(sequences: List[List[Iterable[str]]],
               min_support_abs: int,
               min_pattern_length: int = 1):
    S = [[set(e) for e in seq if e] for seq in sequences if seq]
    if not S:
        return []

    results: List[Tuple[List[Set[str]], int]] = []

    # Itemes frecuentes (aparecen al menos una vez en la secuencia)
    item_support = Counter()
    for seq in S:
        seen = set()
        for ev in seq:
            seen |= ev
        for it in seen:
            item_support[it] += 1
    frequent_items = [it for it, c in item_support.items() if c >= min_support_abs]

    def mine(db: List[List[Set[str]]], prefix: List[Set[str]]):
        # 1) Crecer en el MISMO evento (agregar item al último set)
        if prefix:
            proj_same = _project_db(db, prefix, grow_in_same_event=True)
            cooccur = Counter()
            for seq in proj_same:
                if seq:
                    for it in seq[0]:
                        if it not in prefix[-1]:
                            cooccur[it] += 1
            for it, sup in cooccur.items():
                if sup >= min_support_abs:
                    new_prefix = [set(e) for e in prefix]
                    new_prefix[-1].add(it)
                    if len(new_prefix) >= min_pattern_length:
                        results.append((deepcopy(new_prefix), sup))
                    mine(proj_same, new_prefix)

        # 2) Crecer con NUEVO evento
        proj_next = _project_db(db, prefix, grow_in_same_event=False) if prefix else db
        next_counts = Counter()
        for seq in proj_next:
            if seq:
                for it in seq[0]:
                    next_counts[it] += 1
        if not prefix and not next_counts:
            next_counts = Counter({it: item_support[it] for it in frequent_items})
        for it, sup in next_counts.items():
            if sup >= min_support_abs:
                new_prefix = [set(e) for e in prefix] + [set([it])]
                if len(new_prefix) >= min_pattern_length:
                    results.append((deepcopy(new_prefix), sup))
                mine(proj_next, new_prefix)

    # Semillas: cada item frecuente como evento unitario
    for it in frequent_items:
        sup = item_support[it]
        new_prefix = [set([it])]
        if len(new_prefix) >= min_pattern_length:
            results.append((deepcopy(new_prefix), sup))
        mine(S, new_prefix)

    # Deduplicar
    dedup = {}
    for pat, sup in results:
        key = tuple(tuple(sorted(e)) for e in pat)
        dedup[key] = max(dedup.get(key, 0), sup)

    out = [([set(e) for e in key], sup) for key, sup in dedup.items()]
    out.sort(key=lambda x: (-x[1], x[0]))
    return out

patterns = prefixspan(sequences, min_support_abs=MIN_SUP_SEQUENCES, min_pattern_length=MIN_PATTERN_LENGTH)
print(f"[INFO] Patrones secuenciales encontrados: {len(patterns):,}")

def _pat_to_str(pat: List[Set[str]]) -> str:
    return " -> ".join("{" + ", ".join(sorted(e)) + "}" for e in pat)

display(pd.DataFrame(
    [{"pattern": _pat_to_str(p), "length": len(p), "support": sup}
     for p, sup in patterns[:20]]
))


[INFO] Patrones secuenciales encontrados: 53,539


,pattern,length,support
0,{WHITE HANGING HEART T-LIGHT HOLDER},1,1490
1,{REGENCY CAKESTAND 3 TIER},1,1314
2,{BAKING SET 9 PIECE RETROSPOT},1,1137
3,{ASSORTED COLOUR BIRD ORNAMENT},1,1010
4,{PAPER CHAIN KIT 50'S CHRISTMAS},1,896
5,{PARTY BUNTING},1,894
6,{HEART OF WICKER SMALL},1,886
7,{NATURAL SLATE HEART CHALKBOARD},1,874
8,{JUMBO BAG RED RETROSPOT},1,860
9,{HEART OF WICKER LARGE},1,827
